# Reproduction de la Figure 4A — Score de Maladie

**Référence :** Lundberg & Lee (2017), *A Unified Approach to Interpreting Model Predictions*, NIPS 2017

---

## Contexte

La **Figure 4A** de l'article compare trois méthodes d'attribution de features sur un modèle de score de maladie :
- **SHAP** (méthode proposée par les auteurs)
- **LIME** (méthode existante, basée sur une approximation locale)
- **Intuition humaine** (étude utilisateur via Amazon Mechanical Turk, 30 participants)

### Le modèle

Le **score de maladie** (sickness score) est défini comme suit :

| Fièvre | Toux | Congestion | Score |
|--------|------|------------|-------|
| 0      | 0    | quelconque | **0** |
| 1      | 0    | quelconque | **5** |
| 0      | 1    | quelconque | **5** |
| 1      | 1    | quelconque | **2** |

> **Comportement XOR-like** : Le score est plus élevé quand UN SEUL des deux symptômes (fièvre ou toux) est présent.  
> La congestion n'a **aucun effet** sur le score.

### Question

Un patient a **fièvre=1, toux=1, congestion=1** → score=**2**.  
Quel crédit donner à chaque symptôme pour expliquer ce score de 2 ?

---

### Résultat attendu

- **SHAP et l'intuition humaine** : attribution positive à la fièvre et à la toux  
  → *"Ces deux symptômes augmentent le risque de maladie par rapport à quelqu'un en bonne santé"*
- **LIME** : attribution **négative** à la fièvre et à la toux  
  → *"Localement, avoir les deux symptômes RÉDUIT le score (par rapport à n'en avoir qu'un seul)"*

Ce décalage de LIME montre sa violation de la **cohérence** (*consistency property*, Theorem 1 de l'article).

In [ ]:
import numpy as np
import math
import itertools
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)
print('Imports OK')

## 1. Définition du modèle de score de maladie

In [ ]:
def sickness_score(X):
    """
    Score de maladie (Lundberg & Lee 2017, Figure 4A).

    Input : X, array de shape (n, 3) — colonnes [fièvre, toux, congestion], valeurs binaires {0, 1}
    Output : array de shape (n,) — le score de maladie

    Règles :
      - 0 si ni fièvre ni toux
      - 5 si exactement UN des deux (fièvre OU toux)
      - 2 si les DEUX (fièvre ET toux)
      - la congestion n'a aucun effet
    """
    X = np.atleast_2d(X).astype(float)
    out = np.zeros(X.shape[0])
    for i, row in enumerate(X):
        fever, cough = int(row[0]), int(row[1])
        if fever == 1 and cough == 1:
            out[i] = 2
        elif fever == 1 or cough == 1:
            out[i] = 5
        # else: out[i] = 0 (valeur par défaut)
    return out

feature_names = ['Fièvre', 'Toux', 'Congestion']

# Vérification : toutes les combinaisons possibles
print('Toutes les combinaisons du modèle :')
print(f'{"Fièvre":>8} {"Toux":>6} {"Cong.":>7} → {"Score":>5}')
print('-' * 35)
for fever in [0, 1]:
    for cough in [0, 1]:
        for cong in [0, 1]:
            score = sickness_score([[fever, cough, cong]])[0]
            print(f'{fever:>8} {cough:>6} {cong:>7} → {score:>5.0f}')

## 2. Instance à expliquer

Le patient a **fièvre=1, toux=1, congestion=1** → score = **2**.  
La référence (baseline) est un patient sain : **fièvre=0, toux=0, congestion=0** → score = **0**.  

Les attributions doivent donc sommer à : f(x) − f(ref) = 2 − 0 = **+2**.

In [ ]:
# Instance à expliquer : patient malade avec les deux symptômes
x_instance = np.array([[1., 1., 1.]])

# Référence : patient sain (aucun symptôme)
reference = np.array([[0., 0., 0.]])

score_patient = sickness_score(x_instance)[0]
score_ref     = sickness_score(reference)[0]
diff_to_explain = score_patient - score_ref

print(f'Patient    : fièvre=1, toux=1, congestion=1 → score = {score_patient:.0f}')
print(f'Référence  : fièvre=0, toux=0, congestion=0 → score = {score_ref:.0f}')
print(f'Différence à expliquer : {score_patient:.0f} − {score_ref:.0f} = {diff_to_explain:.0f}')
print()
print('=> Les attributions SHAP doivent sommer à +2')

## 3. Valeurs SHAP — Calcul analytique exact

Les valeurs de Shapley sont définies par (Équation 8 de l'article) :

$$\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(M - |S| - 1)!}{M!} \left[ f(x_{S \cup \{i\}}) - f(x_S) \right]$$

Pour M=3 features, il n'y a que 8 sous-ensembles à évaluer → calcul **exact** possible.

La contribution marginale de la fièvre dans chaque ordering :

| Ordre d'ajout            | Marginale fièvre       |
|--------------------------|------------------------|
| fièvre → toux → cong.   | f(1,0,0)−f(0,0,0)=**+5** |
| fièvre → cong. → toux   | f(1,0,0)−f(0,0,0)=**+5** |
| toux → fièvre → cong.   | f(1,1,0)−f(0,1,0)=**−3** |
| toux → cong. → fièvre   | f(1,1,1)−f(0,1,1)=**−3** |
| cong. → fièvre → toux   | f(1,0,1)−f(0,0,1)=**+5** |
| cong. → toux → fièvre   | f(1,1,1)−f(0,1,1)=**−3** |

Moyenne pondérée : φ_fièvre = (2×5 + 2×5 + 2×(−3) + 2×(−3)) / 6 ... = **(5+5−3−3+5−3)/6 = 6/6 = 1**

In [ ]:
def exact_shapley(f, x, reference, feature_idx, M):
    """
    Calcule exactement la valeur de Shapley pour la feature feature_idx.
    Énumère tous les sous-ensembles S ⊆ F\{i} (2^(M-1) sous-ensembles).
    """
    phi = 0.0
    other_features = [i for i in range(M) if i != feature_idx]

    for r in range(len(other_features) + 1):
        for subset in itertools.combinations(other_features, r):
            subset = list(subset)
            s = len(subset)
            # Poids de Shapley : |S|! * (M - |S| - 1)! / M!
            weight = math.factorial(s) * math.factorial(M - s - 1) / math.factorial(M)

            # f avec la feature i incluse
            x_with = reference.copy()
            for j in subset:
                x_with[0, j] = x[0, j]
            x_with[0, feature_idx] = x[0, feature_idx]

            # f sans la feature i
            x_without = reference.copy()
            for j in subset:
                x_without[0, j] = x[0, j]

            phi += weight * (f(x_with)[0] - f(x_without)[0])

    return phi


M = 3
shap_vals = np.array([
    exact_shapley(sickness_score, x_instance, reference.copy(), i, M)
    for i in range(M)
])

print('Valeurs SHAP (analytique, référence = (0,0,0)) :')
for name, phi in zip(feature_names, shap_vals):
    print(f'  φ_{name:<12} = {phi:+.4f}')
print(f'  {"Somme":<14} = {shap_vals.sum():+.4f}  ✓ (= f(x) − f(ref) = {diff_to_explain:.0f})')
print()
print('Interprétation :')
print('  - La fièvre contribue +1 (positive car elle cause la maladie en général)')
print('  - La toux contribue +1 (idem, même contribution par symétrie)')
print('  - La congestion contribue 0 (n\'a aucun effet dans le modèle)')

### Pourquoi φ_fièvre = +1 (positif) ?

SHAP moyennise sur **tous les ordres possibles** d'introduction des features.

- Quand on ajoute la fièvre **avant** la toux : contribution = f(1,0) − f(0,0) = 5 − 0 = **+5**
- Quand on ajoute la fièvre **après** la toux  : contribution = f(1,1) − f(0,1) = 2 − 5 = **−3**

Moyenne = **(5 − 3) / 2 = +1**

SHAP capture bien que la fièvre est globalement un **facteur de risque** même si, dans ce cas précis (patient avec déjà la toux), son ajout réduit le score.

## 4. Valeurs LIME — Approximation linéaire locale

LIME ajuste un modèle linéaire **localement autour de l'instance** x = (1,1,1).  
Il génère des perturbations binaires z' ∈ {0,1}³, où :
- z'_i = 1 → feature i garde la valeur de l'instance
- z'_i = 0 → feature i prend une valeur background aléatoire

Les échantillons sont **pondérés par un kernel L2** centré sur l'instance :
$$w(z') = \sqrt{\exp\left(-\frac{||z' - \mathbf{1}||^2}{\sigma^2}\right)}$$

avec σ = √M × 0.75 (valeur LIME par défaut).

**Problème clé** : Dans le voisinage immédiat de (1,1,1), supprimer la fièvre fait *augmenter* le score (de 2 à 5). LIME interprète donc la fièvre comme un facteur **négatif** localement, à l'opposé de l'intuition humaine.

In [ ]:
def lime_binary(f, x_instance, background, n_samples=5000, kernel_width=None, seed=0):
    """
    Implémentation manuelle de LIME pour features binaires (tabular).

    Algorithme :
    1. Générer n_samples masques binaires z' ~ Uniform({0,1}^M)
    2. Mapper z' → features réelles (z'_i=1 → x_i, z'_i=0 → valeur background aléatoire)
    3. Évaluer le modèle sur les instances perturbées
    4. Calculer les poids kernel (L2 centré sur l'instance)
    5. Résoudre la régression linéaire pondérée (WLS)
    6. Retourner les coefficients (= attributions LIME)
    """
    rng = np.random.default_rng(seed)
    M = len(x_instance)
    if kernel_width is None:
        kernel_width = np.sqrt(M) * 0.75  # défaut LIME

    # Étape 1 : masques binaires aléatoires
    z_prime = rng.integers(0, 2, size=(n_samples, M)).astype(float)

    # Étape 2 : mapper vers les features réelles
    bg_indices = rng.integers(0, len(background), size=n_samples)
    z_features = background[bg_indices].copy()
    for i in range(M):
        mask_on = z_prime[:, i] == 1
        z_features[mask_on, i] = x_instance[i]

    # Étape 3 : évaluer le modèle
    y = f(z_features)

    # Étape 4 : poids kernel — distance L2 de z' à (1,1,...,1)
    x_prime = np.ones(M)  # représentation binaire de l'instance
    dists = np.sqrt(np.sum((z_prime - x_prime) ** 2, axis=1))
    weights = np.sqrt(np.exp(-(dists ** 2) / (kernel_width ** 2)))

    # Étape 5 : régression linéaire pondérée
    X_design = np.column_stack([np.ones(n_samples), z_prime])
    XtW = X_design.T * weights  # = X' @ diag(W)
    beta, _, _, _ = np.linalg.lstsq(XtW @ X_design, XtW @ y, rcond=None)

    return beta[1:]  # coefficients (hors intercept)


# Background : toutes les 8 combinaisons possibles (distribution uniforme)
background_all = np.array(
    [[i, j, k] for i in [0, 1] for j in [0, 1] for k in [0, 1]], dtype=float
)

# Calcul LIME avec seed=42
lime_vals = lime_binary(sickness_score, x_instance[0], background_all, n_samples=5000, seed=42)

print('Valeurs LIME (seed=42, n=5000) :')
for name, val in zip(feature_names, lime_vals):
    print(f'  φ_{name:<12} = {val:+.4f}')
print()
print('Interprétation problématique :')
print('  - Fièvre et Toux ont des contributions NÉGATIVES selon LIME')
print('  - Localement près de (1,1,1), supprimer un symptôme augmente le score')
print('  - LIME capture le comportement local (inhibition), pas la causalité globale')

### Instabilité de LIME : dépendance à la graine aléatoire

LIME produit des résultats **différents selon la graine aléatoire**. Pour un modèle symétrique (fièvre et toux jouent le même rôle), LIME peut donner des attributions asymétriques selon le tirage — violant la propriété de *consistency* (Propriété 3 de l'article).

In [ ]:
print('Variabilité de LIME selon la graine aléatoire (n_samples=1000) :')
print(f'{"Seed":>6} | {"Fièvre":>8} | {"Toux":>8} | {"Congestion":>11}')
print('-' * 45)
for seed in [0, 1, 2, 3, 10, 42, 100, 123, 999]:
    v = lime_binary(sickness_score, x_instance[0], background_all, n_samples=1000, seed=seed)
    asym = '⚠ asymétrique' if abs(v[0] - v[1]) > 0.15 else ''
    print(f'{seed:>6} | {v[0]:>+8.3f} | {v[1]:>+8.3f} | {v[2]:>+11.3f}  {asym}')
print()
print('=> SHAP est TOUJOURS stable et symétrique : φ_Fièvre = φ_Toux = +1.0')
print('=> LIME peut donner des résultats asymétriques et inconsistants')

## 5. Données humaines (étude utilisateur)

L'article original a recruté **30 participants** via Amazon Mechanical Turk pour attribuer un crédit à chaque feature.

Les données Mechanical Turk originales de la Figure 4A ne sont pas dans ce repo, mais le notebook
`notebooks/tabular_examples/tree_based_models/tree_shap_paper/Figure 3 - User Study.ipynb`
contient des données d'une expérience similaire (34 participants, fièvre/toux).

La réponse la plus commune (11/34) était : **fièvre=30, toux=30** (attribution équivalente aux deux symptômes).  
Normalisée à l'échelle SHAP (somme = f(x) − f(ref) = 2) : **φ_fièvre = 1.0, φ_toux = 1.0**.

Cette attribution humaine est **cohérente avec SHAP** et contraire à LIME.

In [ ]:
# Données humaines du notebook tree_shap_paper (Figure 3 - User Study)
user_results_raw = [
    {"cough": 30, "fever": 30}, {"cough": 20, "fever": 40}, {"cough": 30, "fever": 30},
    {"cough": 30, "fever": 30}, {"cough": 45, "fever": 15}, {"cough": 20, "fever": 40},
    {"cough": 60, "fever": 20}, {"cough": 80, "fever": 20}, {"cough": 40, "fever": 40},
    {"cough": 30, "fever": 30}, {"cough": 20, "fever": 40}, {"cough": 20, "fever": 40},
    {"cough": 10, "fever": 50}, {"cough": 80, "fever": 0},  {"cough": 40, "fever": 40},
    {"cough": 40, "fever": 20}, {"cough": 35, "fever": 25}, {"cough": 30, "fever": 30},
    {"cough": 20, "fever": 40}, {"cough": 30, "fever": 30}, {"cough": 30, "fever": 30},
    {"cough": 30, "fever": 30}, {"cough": 40, "fever": 20}, {"cough": 20, "fever": 40},
    {"cough": 20, "fever": 40}, {"cough": 20, "fever": 40}, {"cough": 30, "fever": 30},
    {"cough": 40, "fever": 20}, {"cough": 30, "fever": 30}, {"cough": 30, "fever": 30},
    {"cough": 35, "fever": 25}, {"cough": 40, "fever": 20}, {"cough": 60, "fever": 20},
    {"cough": 30, "fever": 30},
]

# Trouver la réponse la plus commune
responses = [(r["fever"], r["cough"]) for r in user_results_raw]
from collections import Counter
counts = Counter(responses)
most_common_fever, most_common_cough = counts.most_common(1)[0][0]
total_credit = most_common_fever + most_common_cough

print('Top 5 réponses humaines (fièvre, toux) :')
for (f, c), cnt in counts.most_common(5):
    print(f'  fièvre={f:3}, toux={c:3}  →  {cnt} participants')

# Normaliser à l'échelle SHAP (somme = 2)
human_fever = (most_common_fever / total_credit) * diff_to_explain
human_cough  = (most_common_cough  / total_credit) * diff_to_explain
human_cong   = 0.0  # les participants n'allouaient pas de crédit à la congestion

human_vals = np.array([human_fever, human_cough, human_cong])

print(f'\nRéponse la plus commune : fièvre={most_common_fever}, toux={most_common_cough}')
print(f'Normalisée à l\'échelle SHAP (somme=2) :')
print(f'  Fièvre     = {human_fever:.2f}')
print(f'  Toux       = {human_cough:.2f}')
print(f'  Congestion = {human_cong:.2f}')

## 6. Reproduction de la Figure 4A

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

x_pos = np.arange(len(feature_names))
width = 0.25

# Couleurs inspirées de l'article
color_human = '#aaaaaa'
color_shap  = '#1E88E5'
color_lime  = '#43a047'

bars_h = ax.bar(x_pos - width, human_vals, width, label='Human', color=color_human, zorder=3)
bars_s = ax.bar(x_pos,         shap_vals,  width, label='SHAP',  color=color_shap,  zorder=3)
bars_l = ax.bar(x_pos + width, lime_vals,  width, label='LIME',  color=color_lime,  zorder=3)

# Ligne à zéro
ax.axhline(y=0, color='black', linewidth=0.8)

# Annotations sur les barres
for bars in [bars_h, bars_s, bars_l]:
    for bar in bars:
        h = bar.get_height()
        va = 'bottom' if h >= 0 else 'top'
        offset = 0.03 if h >= 0 else -0.03
        ax.text(bar.get_x() + bar.get_width() / 2, h + offset,
                f'{h:+.2f}', ha='center', va=va, fontsize=8, color='#333333')

ax.set_ylabel('feature attribution value', fontsize=12)
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, fontsize=12)
ax.legend(frameon=False, fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Titre et sous-titre
ax.set_title(
    'Figure 4A — Score de Maladie (reproduction)\n'
    'Patient : fièvre=1, toux=1, congestion=1 → score=2 | Référence : (0,0,0) → score=0',
    fontsize=10, pad=12
)

y_min = min(lime_vals.min(), -0.2) - 0.3
y_max = max(shap_vals.max(), human_vals.max()) + 0.4
ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.savefig('figure4A_sickness_score_reproduction.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : figure4A_sickness_score_reproduction.png')

## 7. Discussion : Pourquoi LIME et SHAP divergent ?

### Comportement local vs global

| Aspect                  | SHAP                                      | LIME                                       |
|-------------------------|-------------------------------------------|--------------------------------------------|
| Perspective             | **Globale** : moyenne sur tous les ordres | **Locale** : voisinage de x=(1,1,1)        |
| Signe de φ_fièvre       | **+1** (positif → cause de maladie)       | **≈−0.65** (négatif → inhibiteur local)    |
| Symétrie fièvre/toux    | ✓ **Garantie** par construction           | ✗ **Non garantie** (dépend du tirage)      |
| Propriété de consistance| ✓ **Satisfaite** (Théorème 1)             | ✗ **Violée** (non unique, instable)        |
| Cohérence avec humains  | ✓ **Oui** (φ = +1 pour les deux)          | ✗ **Non** (attributions négatives)         |

### Intuition clé

LIME ajuste un modèle linéaire **près de (1,1,1)**.  
Dans ce voisinage, quand la toux est déjà présente, passer la fièvre de 0→1 fait baisser le score (de 5 à 2).  
Donc LIME dit : *"localement, la fièvre réduit le score"* → attribution négative.

SHAP, lui, moyennise sur **toutes les situations** :
- Sans toux : la fièvre augmente le score de 0→5 (+5)
- Avec toux  : la fièvre baisse le score de 5→2 (−3)
- Moyenne pondérée = **+1** → la fièvre est globalement un facteur de risque ✓

### Conclusion

SHAP produit des attributions **cohérentes avec l'intuition humaine** car il respecte la propriété de *consistency* (Propriété 3 de l'article) : une feature qui augmente la sortie dans **tous les contextes** ne peut pas avoir une attribution négative. LIME peut violer cette propriété.

In [ ]:
# Résumé récapitulatif
print('='*55)
print('RÉSUMÉ — Figure 4A : Score de Maladie')
print('='*55)
print(f'Patient : fièvre=1, toux=1, congestion=1 → score={score_patient:.0f}')
print(f'Référence: (0,0,0) → score={score_ref:.0f}')
print(f'A expliquer : {diff_to_explain:.0f}')
print()
print(f'{"Feature":<14} {"Human":>8} {"SHAP":>8} {"LIME":>8}')
print('-' * 42)
for name, h, s, l in zip(feature_names, human_vals, shap_vals, lime_vals):
    print(f'{name:<14} {h:>+8.3f} {s:>+8.3f} {l:>+8.3f}')
print('-' * 42)
print(f'{"Somme":<14} {sum(human_vals):>+8.3f} {sum(shap_vals):>+8.3f} {sum(lime_vals):>+8.3f}')
print()
print('Accord avec l\'intuition humaine :')
print('  SHAP : ✓ (valeurs positives, symétriques)')
print('  LIME : ✗ (valeurs négatives, localement correctes mais contre-intuitives)')